# 04 – Quantile Regression Forest (QRF) Modelling

Reproduces Section 4.4 and 5.2:
- Train QRF models (one per LiDAR structural metric) using SAR and
  Sentinel-2 composites as predictors
- 10-fold spatial cross-validation (Section 4.4.1)
- Accuracy assessment: R², RMSE, bias (Table 3 equivalent)
- Variable importance (Figure 12 equivalent)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from src import config
from src.lidar_metrics import METRIC_NAMES

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance
from quantile_forest import RandomForestQuantileRegressor
import joblib

## 4.1 Load training table

In [ ]:
df = pd.read_parquet(config.TRAINING_TABLE)
print(f"Training samples: {len(df):,}")
print(df.head())

In [ ]:
# Define predictor columns (all SAR + S2 composite columns)
predictor_cols = (
    [c for c in df.columns if c.startswith('SAR_')]
    + [c for c in df.columns if c.startswith('s2_')]
)
print(f"Predictors ({len(predictor_cols)}): {predictor_cols}")

## 4.2 Define target metrics to model

In [ ]:
# Reproduce Table 3: model all 23 structural metrics
# For quick testing use a subset:
# TARGET_METRICS = ['GFP', 'PAI', 'stdev', 'p50']
TARGET_METRICS = METRIC_NAMES

## 4.3 10-fold spatial cross-validation + QRF training

In [ ]:
N_FOLDS = 10
QRF_PARAMS = dict(
    n_estimators=500,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
QUANTILES = [0.1, 0.5, 0.9]

results = {}   # metric → {'r2', 'rmse', 'bias', 'pred', 'obs', 'model'}
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

X = df[predictor_cols].values
coords = df[['x', 'y']].values

for metric in TARGET_METRICS:
    y = df[metric].values
    valid = ~np.isnan(y)
    X_v, y_v = X[valid], y[valid]

    pred_median = np.full(y_v.shape[0], np.nan)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_v)):
        model = RandomForestQuantileRegressor(**QRF_PARAMS)
        model.fit(X_v[train_idx], y_v[train_idx])
        pred_median[test_idx] = model.predict(X_v[test_idx], quantiles=0.5)

    r2 = r2_score(y_v, pred_median)
    rmse = np.sqrt(mean_squared_error(y_v, pred_median))
    bias = np.mean(pred_median - y_v)

    # Final model trained on all data
    final_model = RandomForestQuantileRegressor(**QRF_PARAMS)
    final_model.fit(X_v, y_v)

    results[metric] = {
        'r2': r2, 'rmse': rmse, 'bias': bias,
        'pred': pred_median, 'obs': y_v, 'model': final_model,
    }
    print(f"{metric:20s}  R²={r2:.3f}  RMSE={rmse:.4f}  bias={bias:+.4f}")

## 4.4 Accuracy summary table (Table 3 equivalent)

In [ ]:
summary = pd.DataFrame({
    m: {'R²': v['r2'], 'RMSE': v['rmse'], 'Bias': v['bias']}
    for m, v in results.items()
}).T.round(4)
summary.index.name = 'Metric'
summary = summary.sort_values('R²', ascending=False)
print(summary.to_string())
summary.to_csv(config.OUTPUT_DIR / (config.LAS_NAME+'_table_accuracy_summary.csv'))

## 4.5 Observed vs predicted scatter (Figure 11 equivalent)

In [ ]:
show = [m for m in ['GFP', 'PAI', 'stdev', 'dens_total'] if m in results]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 4))
for ax, m in zip(np.atleast_1d(axes), show):
    obs, pred = results[m]['obs'], results[m]['pred']
    ax.scatter(obs, pred, s=4, alpha=0.3, color='steelblue')
    lo, hi = min(obs.min(), pred.min()), max(obs.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1)
    r2 = results[m]['r2']
    rmse = results[m]['rmse']
    ax.set_title(f'{m}\nR²={r2:.3f}, RMSE={rmse:.4f}')
    ax.set_xlabel('Observed')
    ax.set_ylabel('Predicted (median QRF)')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / (config.LAS_NAME+'_fig_obs_vs_pred.png'), dpi=150)
plt.show()

## 4.6 Variable importance (permutation, Figure 12 equivalent)

In [ ]:
imp_metric = 'GFP'   # change as needed
m_res = results[imp_metric]
model = m_res['model']
valid = ~np.isnan(df[imp_metric].values)
X_v, y_v = X[valid], df[imp_metric].values[valid]

perm = permutation_importance(model, X_v, y_v, n_repeats=10, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    'feature': predictor_cols,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df['feature'][:20][::-1], imp_df['importance'][:20][::-1],
        xerr=imp_df['std'][:20][::-1], color='steelblue', capsize=3)
ax.set_xlabel('Permutation importance (decrease in R²)')
ax.set_title(f'Variable importance for predicting {imp_metric}')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / (config.LAS_NAME+'_fig_importance_{imp_metric}.png'), dpi=150)
plt.show()

## 4.7 Save trained models

In [ ]:
model_dir = config.OUTPUT_DIR / 'models'
model_dir.mkdir(exist_ok=True)
for metric, res in results.items():
    safe_name = metric.replace('>', 'gt').replace('<', 'lt')
    joblib.dump(res['model'], model_dir / (config.LAS_NAME+f'qrf_{safe_name}.joblib'))
print(f"Models saved to {model_dir}")